In [1]:
## SETUP & LOAD
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import sys, os

sys.path.insert(0, os.path.abspath('..'))
import config

engine = create_engine(
    f"mysql+pymysql://{config.DB_CONFIG['user']}:{config.DB_CONFIG['password']}"
    f"@{config.DB_CONFIG['host']}:{config.DB_CONFIG['port']}/{config.DB_CONFIG['database']}"
)

# Lightweight loads only
dp = pd.read_sql(
    "SELECT ticker, trade_date, close_price FROM daily_prices ORDER BY ticker, trade_date",
    engine, parse_dates=['trade_date']
)
sf = pd.read_sql(
    "SELECT ticker, daily_return FROM stock_features WHERE daily_return IS NOT NULL",
    engine
)
sf['daily_return'] = pd.to_numeric(sf['daily_return'], downcast='float')

st  = pd.read_sql("SELECT ticker, company_name, market, sector FROM stocks", engine)
vol = pd.read_sql("SELECT ticker, ann_volatility_pct, max_drawdown_pct FROM stock_volatility", engine)

print(f"Loaded: {len(dp):,} price rows | {len(sf):,} return rows | {len(st)} stocks")

Loaded: 289,656 price rows | 289,577 return rows | 80 stocks


In [2]:
## CAGR PER STOCK
# Compound Annual Growth Rate = (end_price / start_price)^(1/years) - 1

dp_sorted = dp.sort_values(['ticker', 'trade_date'])

bounds = dp_sorted.groupby('ticker').agg(
    start_price = ('close_price', 'first'),
    end_price   = ('close_price', 'last'),
    start_date  = ('trade_date',  'min'),
    end_date    = ('trade_date',  'max')
).reset_index()

bounds['years']    = (bounds['end_date'] - bounds['start_date']).dt.days / 365.25
bounds['cagr_pct'] = (
    (bounds['end_price'] / bounds['start_price']) ** (1 / bounds['years']) - 1
) * 100

cagr = bounds[['ticker', 'cagr_pct']].round(2)
print(f"CAGR computed for {len(cagr)} stocks")
print("\nTop 5 by CAGR:")
print(cagr.merge(st[['ticker','company_name','market']], on='ticker')
          .nlargest(5, 'cagr_pct').to_string(index=False))

CAGR computed for 79 stocks

Top 5 by CAGR:
       ticker  cagr_pct  company_name market
         NVDA     47.08        NVIDIA     US
         TSLA     46.80         Tesla     US
BAJFINANCE.NS     43.58 Bajaj Finance  India
         AVGO     41.18      Broadcom     US
         NFLX     37.47       Netflix     US


In [3]:
## SHARPE + CALMAR RATIO
# Risk-free rate: 5% annual (approximate for both markets)
rf_daily = 0.05 / 252

stats = (sf.groupby('ticker')['daily_return']
           .agg(['mean', 'std'])
           .reset_index())
stats.columns = ['ticker', 'mean_return', 'std_return']

# Sharpe = (annualised_return - risk_free) / annualised_volatility
stats['sharpe_ratio'] = (
    (stats['mean_return'] * 252 - 0.05) / (stats['std_return'] * np.sqrt(252))
).round(3)

# Merge with CAGR and max drawdown for Calmar
ratios = stats[['ticker', 'sharpe_ratio']].merge(cagr, on='ticker').merge(
    vol[['ticker', 'max_drawdown_pct']], on='ticker'
)

# Calmar = CAGR / |max drawdown| — higher is better
ratios['calmar_ratio'] = (
    ratios['cagr_pct'] / ratios['max_drawdown_pct'].abs()
).round(3)

print(f"Ratios computed for {len(ratios)} stocks")
print("\nTop 5 by Sharpe Ratio:")
print(ratios.merge(st[['ticker','company_name','market']], on='ticker')
           .nlargest(5, 'sharpe_ratio')
           [['ticker','company_name','market','cagr_pct','sharpe_ratio','calmar_ratio']]
           .to_string(index=False))

Ratios computed for 79 stocks

Top 5 by Sharpe Ratio:
       ticker         company_name market  cagr_pct  sharpe_ratio  calmar_ratio
BAJFINANCE.NS        Bajaj Finance  India     43.58         1.082         0.698
         AVGO             Broadcom     US     41.18         0.987         0.853
         NVDA               NVIDIA     US     47.08         0.966         0.710
 EICHERMOT.NS        Eicher Motors  India     33.98         0.925         0.560
 BRITANNIA.NS Britannia Industries  India     26.76         0.883         0.710


In [4]:
## COMBINE ALL + SAVE TO MYSQL
rr = (ratios.merge(vol[['ticker','ann_volatility_pct']], on='ticker')
            .merge(st, on='ticker'))

rr = rr[['ticker','company_name','market','sector',
          'cagr_pct','sharpe_ratio','calmar_ratio',
          'ann_volatility_pct','max_drawdown_pct']]

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS stock_risk_return (
            rr_id              INT AUTO_INCREMENT PRIMARY KEY,
            ticker             VARCHAR(20) NOT NULL UNIQUE,
            company_name       VARCHAR(100),
            market             ENUM('India','US'),
            sector             VARCHAR(50),
            cagr_pct           DECIMAL(8,2),
            sharpe_ratio       DECIMAL(8,3),
            calmar_ratio       DECIMAL(8,3),
            ann_volatility_pct DECIMAL(8,2),
            max_drawdown_pct   DECIMAL(8,2)
        )
    """))
    conn.execute(text("TRUNCATE TABLE stock_risk_return"))
    conn.commit()

rr.to_sql('stock_risk_return', engine, if_exists='append', index=False)
print(f"Saved {len(rr)} rows to stock_risk_return")
print(rr.groupby('market')[['cagr_pct','sharpe_ratio','calmar_ratio']].mean().round(2))

Saved 79 rows to stock_risk_return
        cagr_pct  sharpe_ratio  calmar_ratio
market                                      
India      16.84          0.51          0.34
US         20.60          0.59          0.46


In [5]:
## CORRELATION MATRIX
sf_full = pd.read_sql(
    "SELECT ticker, trade_date, daily_return FROM stock_features WHERE daily_return IS NOT NULL",
    engine, parse_dates=['trade_date']
)
sf_full['daily_return'] = pd.to_numeric(sf_full['daily_return'], downcast='float')

for market in ['India', 'US']:
    tickers = st[st['market'] == market]['ticker'].tolist()
    pivot   = (sf_full[sf_full['ticker'].isin(tickers)]
               .pivot(index='trade_date', columns='ticker', values='daily_return'))
    corr    = pivot.corr().round(3)

    print(f"\nCorrelation matrix — {market} ({len(tickers)} stocks)")
    print(f"Average pairwise correlation : {corr.values[~np.eye(len(corr), dtype=bool)].mean():.3f}")
    print(f"Highest correlated pair      :", end=' ')

    # Find highest off-diagonal pair
    corr_flat = corr.where(~np.eye(len(corr), dtype=bool))
    max_idx   = corr_flat.stack().idxmax()
    print(f"{max_idx[0]} ↔ {max_idx[1]} = {corr.loc[max_idx]:.3f}")

    # Save matrix to CSV
    corr.to_csv(f"../data/processed/corr_{market.lower()}.csv")
    print(f"Saved → data/processed/corr_{market.lower()}.csv")


Correlation matrix — India (50 stocks)
Average pairwise correlation : 0.258
Highest correlated pair      : JSWSTEEL.NS ↔ TATASTEEL.NS = 0.690
Saved → data/processed/corr_india.csv

Correlation matrix — US (30 stocks)
Average pairwise correlation : 0.376
Highest correlated pair      : BAC ↔ JPM = 0.853
Saved → data/processed/corr_us.csv
